In [ ]:
# Upload the dataset
from google.colab import files

uploaded = files.upload()

In [ ]:
# Import the required libraries
from sklearn.preprocessing import OneHotEncoder
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

In [ ]:
# Reproducibility and GPU check
import random

SEED = 42

def set_seed(seed=SEED):
    """Fix every random source used in this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device     :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU is active. In Colab: Runtime > Change runtime type > "
        "Hardware accelerator = GPU (T4/L4), then restart the session and "
        "run the notebook from the first cell."
    )

In [ ]:
# Load the CSV file into a pandas DataFrame
file_name = 'df_train.csv'
df = pd.read_csv(file_name)

df.info()

## Option A - discrete-value schema

In [ ]:
# OPTION A - DISCRETE-VALUE SCHEMA
import numpy as np
import pandas as pd

SCHEMA_TARGET_COL   = "Target"
MAX_DISCRETE_LEVELS = 12          # at most this many unique values => discrete
ONEHOT_PREFIXES     = ["Urine Test_"]

def build_schema(df_reference, target_col=SCHEMA_TARGET_COL):
    """Describe every feature column of the real training data."""
    schema = {"discrete": {}, "continuous": {}, "onehot_groups": {}}
    features = [c for c in df_reference.columns if c != target_col]

    for prefix in ONEHOT_PREFIXES:
        members = sorted(c for c in features if c.startswith(prefix))
        if members:
            schema["onehot_groups"][prefix] = members

    grouped = {c for m in schema["onehot_groups"].values() for c in m}

    for col in features:
        if col in grouped:
            continue
        levels = np.unique(df_reference[col].dropna().to_numpy(dtype=float))
        if len(levels) <= MAX_DISCRETE_LEVELS:
            schema["discrete"][col] = np.sort(levels)
        else:
            schema["continuous"][col] = (
                float(df_reference[col].min()),
                float(df_reference[col].max()),
            )
    return schema

def apply_schema(df_synth, schema, target_col=SCHEMA_TARGET_COL):
    """Return a copy of df_synth in which every value is legal."""
    out = df_synth.copy()

    # 1. snap each discrete column to its nearest legal level
    for col, levels in schema["discrete"].items():
        if col not in out.columns:
            continue
        values = out[col].to_numpy(dtype=float)
        idx = np.abs(values[:, None] - levels[None, :]).argmin(axis=1)
        out[col] = levels[idx]

    # 2. reduce each one-hot group to a single 1 (arg-max wins)
    for members in schema["onehot_groups"].values():
        members = [c for c in members if c in out.columns]
        if not members:
            continue
        block = out[members].to_numpy(dtype=float)
        hard = np.zeros_like(block)
        hard[np.arange(len(block)), block.argmax(axis=1)] = 1.0
        out[members] = hard

    # 3. keep continuous columns inside the range seen in the real data
    for col, (lo, hi) in schema["continuous"].items():
        if col in out.columns:
            out[col] = out[col].clip(lo, hi)

    return out

def validity_report(df, schema, target_col=SCHEMA_TARGET_COL, label=""):
    """Percentage of rows holding a legal value, per discrete column."""
    rows = []
    for col, levels in schema["discrete"].items():
        if col not in df.columns:
            continue
        ok = np.isin(df[col].to_numpy(dtype=float), levels).mean() * 100.0
        rows.append((col, "discrete", len(levels), round(ok, 2)))

    for prefix, members in schema["onehot_groups"].items():
        members = [c for c in members if c in df.columns]
        if not members:
            continue
        block = df[members].to_numpy(dtype=float)
        ok = (np.isin(block, [0.0, 1.0]).all(axis=1)
              & (block.sum(axis=1) == 1.0)).mean() * 100.0
        rows.append((prefix + "*", "one-hot", len(members), round(ok, 2)))

    rep = pd.DataFrame(rows, columns=["Column", "Type", "Levels", "Valid (%)"])
    rep = rep.sort_values("Valid (%)").reset_index(drop=True)
    print("\n--- Validity report " + str(label) + " ---")
    print("Columns fully valid : {} / {}".format((rep["Valid (%)"] == 100).sum(), len(rep)))
    print("Mean validity       : {:.2f}%".format(rep["Valid (%)"].mean()))
    display(rep)
    return rep

# Build the schema from the real training data
_df_reference = pd.read_csv("df_train.csv")
SCHEMA = build_schema(_df_reference)

print("Discrete columns   :", len(SCHEMA["discrete"]))
print("One-hot groups     :", len(SCHEMA["onehot_groups"]))
print("Continuous columns :", len(SCHEMA["continuous"]))
print("\nDiscrete columns and their legal levels:")
for _c, _l in SCHEMA["discrete"].items():
    print("  {:<32} {}".format(_c, np.round(_l, 4).tolist()))

In [ ]:
df_medgan = df.copy()
TARGET_COL = "Target"

# Separate the features from the target
X = df_medgan.drop(columns=[TARGET_COL])
y = df_medgan[[TARGET_COL]]

# Number of features (required when decoding the generated samples)
n_features = X.shape[1]

# One-hot encoding of the target (scikit-learn >= 1.2)
ohe = OneHotEncoder(sparse_output=False)
y_ohe = ohe.fit_transform(y)

# Concatenate the features and the target
X_np = X.values.astype("float32")  # assumed to be already scaled to [0, 1]
data_medgan = np.hstack([X_np, y_ohe])

# MedGAN input dimension
input_dim = data_medgan.shape[1]

print("MedGAN input prepared")
print("   features       :", n_features)
print("   one-hot target :", y_ohe.shape[1])
print("   input_dim :", input_dim)

In [ ]:
# Main configuration
EPOCHS = 100   #100, 300, 500
TARGET_PER_CLASS = 6000
DEVICE_CONFIG = "GPU"   # or "CPU"

# Torch device
DEVICE = "cuda" if DEVICE_CONFIG == "GPU" and torch.cuda.is_available() else "cpu"

print("Device in use:", DEVICE)

In [ ]:
# MedGAN architecture
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

class Generator(nn.Module):
    def __init__(self, noise_dim=32, latent_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )

    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.net(z)

In [ ]:
# Reset the seed so that weight initialisation is reproducible
set_seed(SEED)

# Training
X_tensor = torch.tensor(data_medgan, dtype=torch.float32).to(DEVICE)

ae = Autoencoder(input_dim).to(DEVICE)
G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)

opt_ae = torch.optim.Adam(ae.parameters(), lr=1e-3)
opt_G = torch.optim.Adam(G.parameters(), lr=1e-4)
opt_D = torch.optim.Adam(D.parameters(), lr=1e-4)

bce = nn.BCELoss()

In [ ]:
for epoch in range(EPOCHS):
    # ===== Autoencoder =====
    recon = ae(X_tensor)
    loss_ae = ((recon - X_tensor) ** 2).mean()

    opt_ae.zero_grad()
    loss_ae.backward()
    opt_ae.step()

    # ===== GAN =====
    z_real = ae.encoder(X_tensor).detach()
    noise = torch.randn(len(z_real), 32).to(DEVICE)
    z_fake = G(noise)

    d_real = D(z_real)
    d_fake = D(z_fake)

    loss_D = bce(d_real, torch.ones_like(d_real)) + \
             bce(d_fake, torch.zeros_like(d_fake))

    opt_D.zero_grad()
    loss_D.backward()
    opt_D.step()

    noise = torch.randn(len(z_real), 32).to(DEVICE)
    z_fake = G(noise)
    loss_G = bce(D(z_fake), torch.ones_like(d_real))

    opt_G.zero_grad()
    loss_G.backward()
    opt_G.step()

    if epoch % 50 == 0:
        print(f"[MedGAN] Epoch {epoch} | AE {loss_ae:.4f} | D {loss_D:.4f} | G {loss_G:.4f}")

In [ ]:
def generate_medgan(n_samples):
    noise = torch.randn(n_samples, 32).to(DEVICE)
    z = G(noise)
    synth = ae.decoder(z).detach().cpu().numpy()
    return synth

In [ ]:
# Reset the seed so that the generated samples are reproducible
set_seed(SEED)

synthetic_data_list = []

unique_targets = sorted(df_medgan[TARGET_COL].unique())
BATCH_SIZE = 50000

for target_class in unique_targets:
    current_count = len(df_medgan[df_medgan[TARGET_COL] == target_class])
    num_to_generate = TARGET_PER_CLASS - current_count

    if num_to_generate <= 0:
        continue

    print(f"Generating {num_to_generate} samples for class {target_class}")

    df_synth_class = pd.DataFrame()
    attempts = 0
    max_attempts = 30

    while len(df_synth_class) < num_to_generate and attempts < max_attempts:
        synth = generate_medgan(BATCH_SIZE)

        X_synth = synth[:, :n_features]
        y_synth_ohe = synth[:, n_features:]
        y_synth = ohe.inverse_transform(y_synth_ohe)

        temp_df = pd.DataFrame(X_synth, columns=X.columns)
        temp_df[TARGET_COL] = y_synth.astype(int)

        temp_df = temp_df[temp_df[TARGET_COL] == target_class]
        df_synth_class = pd.concat([df_synth_class, temp_df], ignore_index=True)
        attempts += 1

    # Truncate if more samples than needed
    if len(df_synth_class) >= num_to_generate:
        df_synth_class = df_synth_class.head(num_to_generate)
        print(f"   The GAN produced the required {num_to_generate} samples")
    else:
        # Fallback: complete the quota with bootstrap resampling of the original records
        shortfall = num_to_generate - len(df_synth_class)
        print(f"   The GAN produced only {len(df_synth_class)}/{num_to_generate}. Adding {shortfall} samples via the fallback.")

        df_class_orig = df_medgan[df_medgan[TARGET_COL] == target_class]
        df_fallback = df_class_orig.sample(n=shortfall, replace=True, random_state=42).copy()

        # Add light Gaussian noise so the resampled records are not exact duplicates
        for col in X.columns:
            noise = np.random.normal(0, 0.005, size=len(df_fallback))
            df_fallback[col] = df_fallback[col] + noise

        df_synth_class = pd.concat([df_synth_class, df_fallback], ignore_index=True)

    synthetic_data_list.append(df_synth_class)

## Option A - restore valid values

In [ ]:
# OPTION A - RESTORE VALID VALUES IN THE SYNTHETIC DATA
df_synthetic = pd.concat(synthetic_data_list, ignore_index=True)

rep_before = validity_report(df_synthetic, SCHEMA, TARGET_COL, "BEFORE correction")

df_synthetic = apply_schema(df_synthetic, SCHEMA, TARGET_COL)

rep_after = validity_report(df_synthetic, SCHEMA, TARGET_COL, "AFTER correction")

# Keep the before/after comparison for the paper
comparison = rep_before.merge(
    rep_after, on=["Column", "Type", "Levels"], suffixes=(" before", " after")
)
display(comparison)

# The saving cell that follows now receives the corrected data
synthetic_data_list = [df_synthetic]

print("\nCorrected synthetic rows: {:,}".format(len(df_synthetic)))

In [ ]:
# BUILD THE COMBINED TRAINING SET
df_augmented = pd.concat([df_medgan, df_synthetic], ignore_index=True)

print("Original training rows :", f"{len(df_medgan):,}")
print("Synthetic rows         :", f"{len(df_synthetic):,}")
print("Combined rows          :", f"{len(df_augmented):,}")

print("\nClass distribution of the combined training set:")
print(df_augmented[TARGET_COL].value_counts().sort_index())

In [ ]:
# STANDARDISED EXPORT
MODEL_TAG = "medgan"
DEVICE_TAG = "gpu" if DEVICE == "cuda" else "cpu"
STEM = f"medgan_{DEVICE_TAG}_epoch{EPOCHS}_augment{TARGET_PER_CLASS}_seed{SEED}"

COMBINED_FILE = f"{STEM}.csv"
SYNTHETIC_ONLY_FILE = f"{STEM}_synthetic_only.csv"

# ---- checks before anything is written -------------------------
assert len(df_medgan) + len(df_synthetic) == len(df_augmented), (
    "Combined frame is not original + synthetic: "
    f"{len(df_medgan)} + {len(df_synthetic)} != {len(df_augmented)}")
assert list(df_augmented.columns) == list(df_synthetic.columns), \
    "Column order differs between the combined and the synthetic frame."
assert df_augmented.isna().sum().sum() == 0, "NaN found in the combined frame."
assert df_synthetic.isna().sum().sum() == 0, "NaN found in the synthetic frame."

counts = df_augmented[TARGET_COL].value_counts().sort_index()

print("Original training rows :", f"{len(df_medgan):,}")
print("Synthetic rows         :", f"{len(df_synthetic):,}")
print("Combined rows          :", f"{len(df_augmented):,}")
print("Per-class counts       :", counts.tolist())

if counts.nunique() == 1:
    print("All classes balanced at", counts.iloc[0])
else:
    print("WARNING - the classes are not balanced.")

df_augmented.to_csv(COMBINED_FILE, index=False)
df_synthetic.to_csv(SYNTHETIC_ONLY_FILE, index=False)

print("\nSaved:")
print("  ", COMBINED_FILE)
print("       scenario 2 - df_train + synthetic")
print("  ", SYNTHETIC_ONLY_FILE)
print("       scenario 3 - synthetic only, and the input for notebook 06")

files.download(COMBINED_FILE)
files.download(SYNTHETIC_ONLY_FILE)